# Medical Entity Extraction

**NLP IE — Extract symptoms, drugs, diagnoses from clinical notes**

## Project Overview

Build a medical NER pipeline using **spaCy EntityRuler** to extract
**symptoms**, **drugs**, and **diagnoses** from clinical notes.

## Learning Objectives

1. Apply NER to medical text.
2. Build domain-specific entity patterns.
3. Handle medical terminology.
4. Understand HIPAA considerations.

## Problem Statement

Extract structured medical entities from clinical notes.

## Why This Project Matters

- Clinical notes contain critical information.
- Automated extraction supports pharmacovigilance.
- Medical NER is foundational healthcare NLP.

## Dataset Overview

10 synthetic clinical notes.

## Dataset Source and License Notes

All notes are synthetic. No real patient data.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable,"-m","spacy","download","en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
SYMPTOMS = ["fever","headache","cough","chest pain","fatigue","nausea",
            "dizziness","shortness of breath","back pain","insomnia",
            "anxiety","depression","vomiting","sore throat","joint pain","rash"]
DRUGS = ["ibuprofen","aspirin","metformin","lisinopril","omeprazole",
         "amoxicillin","acetaminophen","prednisone","atorvastatin",
         "losartan","insulin","albuterol","gabapentin","sertraline","amlodipine"]
DIAGNOSES = ["diabetes","hypertension","pneumonia","asthma","bronchitis",
             "arthritis","migraine","anemia","COPD","heart failure",
             "COVID-19","influenza","anxiety disorder","major depressive disorder"]
print(f"Symptoms: {len(SYMPTOMS)}, Drugs: {len(DRUGS)}, Diagnoses: {len(DIAGNOSES)}")


## Dataset Download or Loading

In [ ]:
NOTES = [
    "Patient presents with fever and cough for 3 days. Prescribed amoxicillin. Diagnosis: pneumonia.",
    "Chronic headache and fatigue. History of hypertension. Taking lisinopril and aspirin.",
    "Chest pain and shortness of breath. Rule out heart failure. Started losartan.",
    "Diabetes management. Adjusted metformin. Patient reports nausea.",
    "Follow-up for asthma. Improved on albuterol. Occasional cough persists.",
    "Anxiety and insomnia. Started sertraline. Sleep hygiene discussed.",
    "Acute bronchitis with cough and sore throat. Prescribed acetaminophen.",
    "Hypertension follow-up. BP 145/92. Added amlodipine to lisinopril.",
    "Joint pain and fatigue. Arthritis suspected. Started ibuprofen. Labs for anemia.",
    "Depression screening positive. Major depressive disorder. Gabapentin for back pain.",
]
df = pd.DataFrame({"text": NOTES})
print(f"Loaded {len(df)} clinical notes")


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
print(f"Validated: {len(df)} notes")


## Exploratory Data Analysis

In [ ]:
df["words"] = df["text"].str.split().apply(len)
print(f"Avg words: {df["words"].mean():.0f}")


## Target Analysis

Entity mentions: symptoms, drugs, diagnoses.

## Train/Validation/Test Split Strategy

Not applicable — rule-based.

## Preprocessing Strategy

spaCy EntityRuler with medical patterns.

## Feature Engineering — Build Medical NER

In [ ]:
nlp = spacy.load("en_core_web_sm")
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = ([{"label":"SYMPTOM","pattern":s} for s in SYMPTOMS]
    + [{"label":"DRUG","pattern":d} for d in DRUGS]
    + [{"label":"DIAGNOSIS","pattern":d} for d in DIAGNOSES])
ruler.add_patterns(patterns)
print(f"Added {len(patterns)} medical patterns")


## Baseline Model — Extract Entities

In [ ]:
all_entities = []
for i, note in enumerate(NOTES):
    doc = nlp(note)
    ents = defaultdict(list)
    for e in doc.ents:
        if e.label_ in ("SYMPTOM","DRUG","DIAGNOSIS"): ents[e.label_].append(e.text)
    all_entities.append(dict(ents))
    print(f"\nNote {i+1}: {note[:50]}...")
    for l in ["SYMPTOM","DRUG","DIAGNOSIS"]:
        if ents[l]: print(f"  {l}: {ents[l]}")


## LazyPredict Benchmark

> **Not applicable.** NLP IE task.

## FLAML AutoML Run

> **Not applicable.** FLAML not used for NLP IE.

## Additional Best-Library Workflow — Frequency Analysis

In [ ]:
counts = defaultdict(lambda: defaultdict(int))
for ents in all_entities:
    for label, vals in ents.items():
        for v in vals: counts[label][v] += 1
fig, axes = plt.subplots(1,3,figsize=(16,4))
for ax, label in zip(axes, ["SYMPTOM","DRUG","DIAGNOSIS"]):
    c = dict(sorted(counts[label].items(), key=lambda x:-x[1])[:8])
    if c: ax.barh(list(c.keys()), list(c.values()), color="steelblue"); ax.invert_yaxis()
    ax.set_title(f"{label} Frequency")
plt.tight_layout(); plt.show()


## Top 3 Model Selection

Single pipeline.

In [ ]:
total = {l: sum(c.values()) for l, c in counts.items()}
for l, n in total.items(): print(f"  {l}: {n}")


## Final Training and Evaluation of Top 3

EntityRuler pipeline is the final system.

## Error Analysis

In [ ]:
for i, e in enumerate(all_entities):
    if not e: print(f"Note {i+1}: NO entities")
print("Error analysis complete.")


## Interpretation and Business Insight

1. Rule-based NER catches listed medical terms.
2. Domain dictionaries are essential.
3. For production, use SciSpacy or BioBERT.
4. Negation detection is critical.

## Limitations

- Not exhaustive.
- No negation detection.
- No dosage extraction.
- English-only.

## How to Improve This Project

1. Use SciSpacy or BioBERT.
2. Add negation detection.
3. Extract dosage/frequency.
4. Map to SNOMED-CT/ICD-10.

## Production Considerations

- HIPAA compliance mandatory.
- De-identification before NLP.
- Clinical validation required.
- HL7/FHIR integration.

## Common Mistakes

1. Ignoring negation.
2. Not handling abbreviations.
3. Processing PHI without safeguards.

## Mini Challenge / Exercises

1. Add DOSAGE entity type.
2. Implement negation detection.
3. Map to ICD-10 codes.

## Final Summary / Key Takeaways

1. Medical entity extraction is high-impact.
2. Custom patterns catch domain terms.
3. Negation detection is critical.
4. HIPAA compliance is non-negotiable.
5. For production, use biomedical NER models.